In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from utils import *
import re
import random

In [ ]:
train = pd.read_csv("data/train.csv")

In [ ]:


def show_n_lyrics(df,n,random_state = None):

    """
    Function to show a randomly sampled subset of songs
    """

    rdm = random.Random(random_state)
    indexes =  rdm.sample(range(0, len(df)+1), n)
    counter = 1
    for index in indexes:

        print(f"Song number: {counter}")
        print(ascii(df["lyrics"].iloc[index]))
        print("-"*10)
        counter += 1

In [ ]:
show_n_lyrics(train,10,random_state=77)

In [ ]:
#Dropping all columns except "tag" and "lyrics", because the genre prediction will be
#based solely on the lyrics
train = train.drop(["title","year","features","artist","id","views"],axis=1)

In [ ]:
def search_non_alphanumeric(df):
    """this is a function made with the purpose of
    looking for emojis, unicode characters and other
    characters that are not alphanumeric
    """
    non_alphanumeric = []
    for lyric in df["lyrics"]:
        non_alphanumeric.extend(re.findall("\W",lyric))

    non_alphanumeric = set(non_alphanumeric)
    print(non_alphanumeric)

In [ ]:
search_non_alphanumeric(train)

In [ ]:
teste = train.copy()

In [ ]:

def search_boiler_plate(df,patterns):

    """
    Detects the presence of any Genius boiler plate tags 
    in any song
    prints Y if 1 more songs contain the regex pattern being searched
    prints N if 0 songs contain the regex pattern being searched
    """

    for pattern in patterns:
        flag = False
        for lyric in df["lyrics"]:
            if re.findall(pattern, lyric):
                flag = True
                print(f"{pattern} : Y")
                break
        
        if not flag:
            print(f"{pattern} : N")
        

In [ ]:
search_boiler_plate(train,[
    r"(?im)^\d+\s*Contributors?.*$",         
    r"(?m)^.*?Lyrics\s*$",                     
    r"(?i)\bEmbed\b",                          
    r"(?i)You might also like",
    r"(?m)^\d+$", 
    r"(?im)^(verse|chorus|intro|outro|bridge|pre-chorus|hook|interlude)\b.*?:",
    r"\[.*?\]",
    r"(?i)\(\s*x\s*\d+\s*\)",
    r"\[\d{2}:\d{2}(?:\.\d{2,3})?\]"
])

In [ ]:
teste["lyrics"] = teste["lyrics"].apply(preprocess_text)

In [ ]:
show_n_lyrics(teste,10,random_state=77)

In [ ]:
import torch

print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

In [ ]:
import numpy as np
import torch
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

def get_stopword_ratio(text):
    words = str(text).split()
    return sum(1 for word in words if word in stop_words) / len(words) if len(words) > 0 else 0.0

def get_total_words(text):
    return len(str(text).split())

def get_unique_words(text):
    return len(set(str(text).split()))

# 1. Compute Tabular Features on the already cleaned 'teste' dataframe
teste['stopword_ratio'] = teste['lyrics'].apply(get_stopword_ratio)
teste['word_count'] = teste['lyrics'].apply(get_total_words)
teste['unique_words'] = teste['lyrics'].apply(get_unique_words)

# 2. Scale Features
scaler = StandardScaler()
meta_cols = ['stopword_ratio', 'word_count', 'unique_words']
X_meta = scaler.fit_transform(teste[meta_cols])

# 3. Encode Target (Forced to 1D integer array to prevent stratify bugs)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(teste['tag']).astype(int).ravel()

# 4. Split Text and Meta-Features (Using .tolist() instead of .values to prevent NumPy indexing crashes)
texts_train, texts_val, X_meta_train, X_meta_val, y_train, y_val = train_test_split(
    teste['lyrics'].tolist(), X_meta, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Handle Heavy Class Imbalance (Using torch.from_numpy to prevent PyTorch type inference errors)
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights_tensor = torch.from_numpy(class_weights).float()

In [ ]:
from sklearn.metrics import f1_score
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing on: {device}") # Double check this says 'cuda'!

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Pre-tokenizing data (this takes a few seconds)...")
# Tokenizing everything upfront removes the CPU bottleneck
train_encodings = tokenizer(texts_train, truncation=True, padding='max_length', max_length=256)
val_encodings = tokenizer(texts_val, truncation=True, padding='max_length', max_length=256)
print("Tokenization complete!")

# 1. Optimized Dataset
class FastLyricsDataset(Dataset):
    def __init__(self, encodings, meta, labels=None):
        self.encodings = encodings
        self.meta = meta
        self.labels = labels

    def __len__(self):
        return len(self.meta)

    def __getitem__(self, idx):
        # Extract pre-calculated tensors instantly
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['meta'] = torch.tensor(self.meta[idx], dtype=torch.float)
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# 2. Dual-Input Model Definition
class DistilBertWithMeta(nn.Module):
    def __init__(self, num_classes, meta_dim=3, dropout_rate=0.6):
        super().__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        hidden_size = self.bert.config.hidden_size 
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_size + meta_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, num_classes)
        )

    def forward(self, input_ids, attention_mask, meta):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_token = outputs.last_hidden_state[:, 0, :] 
        combined = torch.cat([cls_token, meta], dim=1) 
        return self.classifier(combined)

# 3. Training Loop
train_loader = DataLoader(FastLyricsDataset(train_encodings, X_meta_train, y_train), batch_size=16, shuffle=True)
val_loader = DataLoader(FastLyricsDataset(val_encodings, X_meta_val, y_val), batch_size=16, shuffle=False)

model = DistilBertWithMeta(num_classes=len(label_encoder.classes_)).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)

EPOCHS = 3
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        logits = model(
            batch['input_ids'].to(device),
            batch['attention_mask'].to(device),
            batch['meta'].to(device)
        )
        loss = criterion(logits, batch['labels'].to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Validation Phase
    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for batch in val_loader:
            logits = model(
                batch['input_ids'].to(device),
                batch['attention_mask'].to(device),
                batch['meta'].to(device)
            )
            val_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            val_targets.extend(batch['labels'].to('cpu').numpy())

    val_f1 = f1_score(val_targets, val_preds, average='weighted')
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {total_loss/len(train_loader):.4f} | Val F1: {val_f1:.4f}")

In [ ]:
import pandas as pd
from torch.utils.data import DataLoader
import torch

test = pd.read_csv("data/test.csv")
submission = pd.read_csv("data/sample_submission.csv")

# 1. Clean test lyrics if you have your preprocess function loaded
if 'preprocess_text' in globals():
    test['lyrics_clean'] = test['lyrics'].apply(preprocess_text)
else:
    test['lyrics_clean'] = test['lyrics']

# 2. Build test tabular features
test['stopword_ratio'] = test['lyrics_clean'].apply(get_stopword_ratio)
test['word_count'] = test['lyrics_clean'].apply(get_total_words)
test['unique_words'] = test['lyrics_clean'].apply(get_unique_words)

# 3. Scale using the already fitted scaler
X_test_meta = scaler.transform(test[['stopword_ratio', 'word_count', 'unique_words']])

# 4. Pre-tokenize the test text (Required for FastLyricsDataset)
print("Pre-tokenizing test data...")
test_encodings = tokenizer(test['lyrics_clean'].tolist(), truncation=True, padding='max_length', max_length=256)

# 5. Dataloader & Inference (Using the updated FastLyricsDataset)
test_loader = DataLoader(FastLyricsDataset(test_encodings, X_test_meta, labels=None), batch_size=16, shuffle=False)

model.eval()
test_preds = []
with torch.no_grad():
    for batch in test_loader:
        logits = model(
            batch['input_ids'].to(device),
            batch['attention_mask'].to(device),
            batch['meta'].to(device)
        )
        test_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())

# 6. Map back to string tags and merge on ID to preserve Kaggle's required row order
test['predicted_tag'] = label_encoder.inverse_transform(test_preds)

final_sub = submission[['id']].merge(test[['id', 'predicted_tag']], on='id', how='left')
final_sub = final_sub.rename(columns={'predicted_tag': 'tag'})

final_sub.to_csv("distilbert_submission.csv", index=False)
print("Transformer submission saved successfully!")